In [273]:
import numpy as np
import pandas as pd
import xarray as xr
from itertools import product

In [274]:
# Load probability table (trained on data <= 2024)
final_path='../data/processed/SPY_cond_ex_tab.nc'
ds = xr.open_dataset(final_path, engine='netcdf4')

# Load full price data
df = pd.read_csv(
    '../data/raw/SPY_daily_yahoo_raw.csv',
    skiprows=[0,1,2],
    header=None,
    names=['Date','Adj Close','Close','High','Low','Open','Volume'],
    index_col=0,
    parse_dates=True
)

df = df.sort_index()
prices = df['Adj Close']

In [275]:
prices_2025 = prices.loc['2025-01-01':'2025-12-31']

In [276]:
directions = ["+", "-"]
q_grid = ds.q.values
r_grid = ds.r.values
q_vals = [q for q in q_grid if 0.8 <= q <= 0.98 and abs(q*10 - round(q*10)) < 1e-8]
r_vals = [r for r in r_grid if r in (0.8, 0.85, 0.9, 0.95)]
taus = [100, 200, 300]
p_mins = [0.5]
se_maxes = [0.01]

In [277]:
quantile_thresholds = {}

P = prices.values

for tau in taus:
    inc = P[tau:] - P[:-tau]
    for q in q_vals:
        quantile_thresholds[(tau, q, "+")] = np.quantile(inc, q)
        quantile_thresholds[(tau, q, "-")] = np.quantile(inc, 1 - q)

In [278]:
results = []

price_array = prices.values
date_index = prices.index

for q, r, tau, p_min, se_max in product(
    q_vals, r_vals, taus, p_mins, se_maxes
):

    # --- lookup probability & uncertainty (PLUS ONLY) ---
    p_hat = ds.prob.sel(direction="+", tau=tau, q=q, r=r).item()
    p_se  = ds.prob_se.sel(direction="+", tau=tau, q=q, r=r).item()

    # --- eligibility ---
    if np.isnan(p_hat) or p_hat < p_min or p_se > se_max:
        results.append({
            "q": q,
            "r": r,
            "tau": tau,
            "p_min": p_min,
            "se_max": se_max,
            "trades": 0,
            "total_return_pct": 0.0,
            "sharpe": np.nan
        })
        continue

    trade_pnls = []

    # --- loop over all days ---
    for date in prices_2025.index:

        idx = date_index.get_loc(date)

        if idx - tau < 0 or idx + tau >= len(price_array):
            continue

        # --- signal ---
        inc_now = price_array[idx] - price_array[idx - tau]
        thresh = quantile_thresholds[(tau, q, "+")]

        if inc_now < thresh:
            continue

        # --- trade P&L ---
        entry_price = price_array[idx]
        exit_price  = price_array[idx + tau]

        pnl = exit_price - entry_price   # raw profit in price units
        trade_pnls.append(pnl)

    # --- compute net return on invested capital ---
    if len(trade_pnls) == 0:
        total_return_pct = 0.0
        sharpe = np.nan
    else:
        # assume 1 unit invested per trade
        total_return = sum(trade_pnls) / price_array[date_index.get_loc(prices_2025.index[0])]
        total_return_pct = 100 * total_return

        trade_returns = np.array(trade_pnls) / price_array[date_index.get_loc(prices_2025.index[0])]

        if trade_returns.std(ddof=1) > 0:
            sharpe = (trade_returns.mean() / trade_returns.std(ddof=1)) * np.sqrt(252 / tau)
        else:
            sharpe = np.nan

    results.append({
        "q": q,
        "r": r,
        "tau": tau,
        "p_min": p_min,
        "se_max": se_max,
        "trades": len(trade_pnls),
        "total_return_pct": total_return_pct,
        "sharpe": sharpe
    })


In [279]:
results_df = pd.DataFrame(results)

In [280]:
results_df.sort_values("total_return_pct", ascending=False).head(10)

,q,r,tau,p_min,se_max,trades,total_return_pct,sharpe
1,0.8,0.8,200,0.5,0.01,55,902.588544,4.618535
4,0.9,0.8,200,0.5,0.01,44,653.156118,6.320112
3,0.9,0.8,100,0.5,0.01,51,245.721713,1.894747
0,0.8,0.8,100,0.5,0.01,0,0.000000,NaN
2,0.8,0.8,300,0.5,0.01,0,0.000000,NaN
5,0.9,0.8,300,0.5,0.01,0,0.000000,NaN


In [281]:
results_df.sort_values("sharpe", ascending=False).head(10)

,q,r,tau,p_min,se_max,trades,total_return_pct,sharpe
4,0.9,0.8,200,0.5,0.01,44,653.156118,6.320112
1,0.8,0.8,200,0.5,0.01,55,902.588544,4.618535
3,0.9,0.8,100,0.5,0.01,51,245.721713,1.894747
0,0.8,0.8,100,0.5,0.01,0,0.000000,NaN
2,0.8,0.8,300,0.5,0.01,0,0.000000,NaN
5,0.9,0.8,300,0.5,0.01,0,0.000000,NaN


In [282]:
ds.close()